In [36]:
# =====================================================================
# SEL 1: IMPORT LIBRARY DAN LOAD DATA READY-MODELING (FIXED PATH)
# =====================================================================
import os
import pandas as pd
from pycaret.regression import setup, compare_models, pull, set_config

# MENGGUNAKAN ABSOLUTE PATH 
file_path = r"d:\lentera-laut\src\data\processed\modeling_ready.csv"

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"✅ Data berhasil dimuat. Total siap modeling: {len(df)} baris.")
else:
    print(f"❌ File TETAP tidak ditemukan di: {file_path}")
    print("Yuk cek tips di bawah untuk memastikan letak filenya!")

df.head()

✅ Data berhasil dimuat. Total siap modeling: 4125 baris.


,time,location,wave_height,wind_speed_10m,precipitation,visibility,ocean_current_velocity,sea_surface_temperature,wave_height_lag1,wave_height_lag2,...,precipitation_lag3,visibility_lag1,visibility_lag2,visibility_lag3,ocean_current_velocity_lag1,ocean_current_velocity_lag2,ocean_current_velocity_lag3,sea_surface_temperature_lag1,sea_surface_temperature_lag2,sea_surface_temperature_lag3
0,2026-05-20 03:00:00,Bangkalan,0.04,8.5,0.0,11680.0,2.2,31.4,0.04,0.04,...,0.0,11680.0,14100.0,14100.0,2.4,1.8,0.9,31.4,31.5,31.5
1,2026-05-20 04:00:00,Bangkalan,0.04,8.7,0.0,11680.0,1.7,31.4,0.04,0.04,...,0.0,11680.0,11680.0,14100.0,2.2,2.4,1.8,31.4,31.4,31.5
2,2026-05-20 05:00:00,Bangkalan,0.04,8.7,0.0,11680.0,0.9,31.3,0.04,0.04,...,0.0,11680.0,11680.0,11680.0,1.7,2.2,2.4,31.4,31.4,31.4
3,2026-05-20 06:00:00,Bangkalan,0.04,9.1,0.0,11680.0,0.3,31.3,0.04,0.04,...,0.0,11680.0,11680.0,11680.0,0.9,1.7,2.2,31.3,31.4,31.4
4,2026-05-20 07:00:00,Bangkalan,0.04,6.6,0.0,14100.0,0.6,31.3,0.04,0.04,...,0.0,11680.0,11680.0,11680.0,0.3,0.9,1.7,31.3,31.3,31.4


In [37]:
# =====================================================================
# SEL 2: MODELING LOOP & COMPARISON (TOP 3 PER TARGET VARIABLE)
# =====================================================================
# Menghapus fitur kronologis agar tidak memicu data leakage pada algoritma regresi standar
model_df = df.drop(columns=["time", "location"])

# Inisialisasi parameter eksperimen
all_results = {}
features = [
    "wave_height", "wind_speed_10m", "ocean_current_velocity", 
    "sea_surface_temperature", "precipitation", "visibility"
]

for target in features:
    print(f"\n{'='*20} PROSES SELECTION TARGET: {target} {'='*20}")
    
    # Inisialisasi environment PyCaret secara asinkron/sinkron
    s = setup(
        data=model_df,
        target=target,
        session_id=123,
        fold=5,
        train_size=0.8,
        verbose=False,
        log_experiment=False, # Set ke False jika tidak memakai MLflow agar runtime lebih cepat
        experiment_name=f"LenteraLaut_Selection_{target}"
    )
    
    # Membandingkan seluruh arsitektur model berdasarkan metrik R-Squared (R2)
    best_models = compare_models(n_select=3, sort="R2")
    
    # Ekstraksi papan peringkat (leaderboard) metrik
    results = pull()
    all_results[target] = {"best_models": best_models, "results": results}
    
    print(f"Top 3 Model Terbaik untuk {target}:")
    print(results.head(3)[["Model", "MAE", "RMSE", "R2"]])

# --- KOMPILASI RINGKASAN TOP 3 MODEL ---
compiled_rows = []
for target, data in all_results.items():
    top_3 = data["results"].head(3).copy()
    top_3.insert(0, "Target Variable", target)
    compiled_rows.append(top_3)

pycaret_summary_df = pd.concat(compiled_rows, ignore_index=True)

# Format visualisasi dataframe untuk laporan
styled_summary = pycaret_summary_df.style.background_gradient(
    subset=["R2"], cmap="YlGn"
).background_gradient(
    subset=["MAE", "RMSE"], cmap="Reds"
).format({
    "R2": "{:.4f}", "MAE": "{:.4f}", "RMSE": "{:.4f}", "MSE": "{:.4f}", "MAPE": "{:.4f}"
})

print("\n" + "="*50)
print("--- TABEL PERBANDINGAN TOP 3 MODEL PYCARET UNTUK SETIAP TARGET ---")
print("="*50)
display(styled_summary)


==================== PROSES SELECTION TARGET: wave_height ====================


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lr,Linear Regression,0.0065,0.0001,0.0099,0.9997,0.0071,0.0349,2.2700
br,Bayesian Ridge,0.0066,0.0001,0.0099,0.9997,0.0071,0.0351,0.0200
gbr,Gradient Boosting Regressor,0.0069,0.0001,0.0111,0.9996,0.0077,0.0352,0.1820
lightgbm,Light Gradient Boosting Machine,0.0072,0.0001,0.0110,0.9996,0.0074,0.0359,0.1180
et,Extra Trees Regressor,0.0071,0.0001,0.0112,0.9996,0.0077,0.0344,0.1480
rf,Random Forest Regressor,0.0074,0.0001,0.0114,0.9996,0.0079,0.0361,0.2840
xgboost,Extreme Gradient Boosting,0.0081,0.0001,0.0118,0.9995,0.0080,0.0412,2.0880
ridge,Ridge Regression,0.0093,0.0002,0.0138,0.9993,0.0097,0.0470,0.6020
dt,Decision Tree Regressor,0.0088,0.0002,0.0153,0.9992,0.0104,0.0420,0.0160
ada,AdaBoost Regressor,0.0215,0.0008,0.0275,0.9974,0.0221,0.2245,0.0940


Top 3 Model Terbaik untuk wave_height:
                           Model     MAE    RMSE      R2
lr             Linear Regression  0.0065  0.0099  0.9997
br                Bayesian Ridge  0.0066  0.0099  0.9997
gbr  Gradient Boosting Regressor  0.0069  0.0111  0.9996

==================== PROSES SELECTION TARGET: wind_speed_10m ====================


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,1.1553,2.5108,1.5843,0.8567,0.2695,0.3082,0.1140
rf,Random Forest Regressor,1.1607,2.5581,1.5990,0.8540,0.2705,0.3111,0.4060
et,Extra Trees Regressor,1.1559,2.5908,1.6091,0.8520,0.2730,0.3128,0.2200
lr,Linear Regression,1.1630,2.5913,1.6091,0.8518,0.2809,0.3002,0.0140
ridge,Ridge Regression,1.1626,2.5928,1.6095,0.8517,0.2807,0.2999,0.0100
br,Bayesian Ridge,1.1642,2.5991,1.6114,0.8514,0.2807,0.3001,0.0080
gbr,Gradient Boosting Regressor,1.1861,2.6092,1.6146,0.8508,0.2749,0.3264,0.1800
xgboost,Extreme Gradient Boosting,1.1985,2.6921,1.6404,0.8464,0.2800,0.3152,0.0860
lar,Least Angle Regression,1.1876,2.7129,1.6466,0.8450,0.2853,0.3028,0.0120
huber,Huber Regressor,1.2569,3.0352,1.7405,0.8261,0.3038,0.3196,0.0180


Top 3 Model Terbaik untuk wind_speed_10m:
                                    Model     MAE    RMSE      R2
lightgbm  Light Gradient Boosting Machine  1.1553  1.5843  0.8567
rf                Random Forest Regressor  1.1607  1.5990  0.8540
et                  Extra Trees Regressor  1.1559  1.6091  0.8520

==================== PROSES SELECTION TARGET: ocean_current_velocity ====================


...

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
gbr,Gradient Boosting Regressor,0.1241,0.0784,0.2796,0.4957,0.1510,0.6569,0.2120
et,Extra Trees Regressor,0.1192,0.0799,0.2826,0.4890,0.1533,0.7118,0.1500
lightgbm,Light Gradient Boosting Machine,0.1257,0.0828,0.2872,0.4732,0.1561,0.7184,0.1000
rf,Random Forest Regressor,0.1206,0.0834,0.2887,0.4645,0.1566,0.7316,0.3540
xgboost,Extreme Gradient Boosting,0.1277,0.0849,0.2910,0.4611,0.1580,0.7434,0.0820
ridge,Ridge Regression,0.1454,0.0890,0.2977,0.4324,0.1640,0.7011,0.0220
br,Bayesian Ridge,0.1452,0.0890,0.2977,0.4323,0.1639,0.6960,0.0160
lr,Linear Regression,0.1458,0.0890,0.2978,0.4320,0.1642,0.7079,0.0260
knn,K Neighbors Regressor,0.1731,0.1375,0.3690,0.1407,0.2142,0.7720,0.0180
llar,Lasso Least Angle Regression,0.2072,0.1481,0.3839,0.0687,0.2223,0.6335,0.0160


Top 3 Model Terbaik untuk precipitation:
                                    Model     MAE    RMSE      R2
gbr           Gradient Boosting Regressor  0.1241  0.2796  0.4957
et                  Extra Trees Regressor  0.1192  0.2826  0.4890
lightgbm  Light Gradient Boosting Machine  0.1257  0.2872  0.4732

==================== PROSES SELECTION TARGET: visibility ====================


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
rf,Random Forest Regressor,1975.6062,8948334.9162,2987.3051,0.7085,0.5130,0.4744,0.4180
et,Extra Trees Regressor,1981.8156,9058078.8415,3006.6557,0.7054,0.5200,0.4874,0.2180
lightgbm,Light Gradient Boosting Machine,2024.9012,9155271.7914,3024.0176,0.7018,0.5366,0.4903,0.1000
gbr,Gradient Boosting Regressor,2076.5808,9258319.8112,3039.8431,0.6987,0.5481,0.5452,0.1820
xgboost,Extreme Gradient Boosting,2076.5201,9616721.4000,3099.0319,0.6867,0.5720,0.5349,0.1020
lasso,Lasso Regression,2250.4138,10365747.2000,3216.5938,0.6634,0.6409,0.7218,0.0180
llar,Lasso Least Angle Regression,2250.5039,10366617.0000,3216.7330,0.6633,0.6413,0.7220,0.0120
ridge,Ridge Regression,2252.1661,10372159.6000,3217.6049,0.6631,0.6407,0.7214,0.0160
lr,Linear Regression,2255.1176,10390345.2000,3220.5301,0.6625,0.6402,0.7187,0.0160
lar,Least Angle Regression,2279.8776,10540516.6000,3244.0189,0.6578,0.6419,0.7264,0.0140


Top 3 Model Terbaik untuk visibility:
                                    Model        MAE       RMSE      R2
rf                Random Forest Regressor  1975.6062  2987.3051  0.7085
et                  Extra Trees Regressor  1981.8156  3006.6557  0.7054
lightgbm  Light Gradient Boosting Machine  2024.9012  3024.0176  0.7018

--- TABEL PERBANDINGAN TOP 3 MODEL PYCARET UNTUK SETIAP TARGET ---


,Target Variable,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
0,wave_height,Linear Regression,0.0065,0.0001,0.0099,0.9997,0.007100,0.0349,2.270000
1,wave_height,Bayesian Ridge,0.0066,0.0001,0.0099,0.9997,0.007100,0.0351,0.020000
2,wave_height,Gradient Boosting Regressor,0.0069,0.0001,0.0111,0.9996,0.007700,0.0352,0.182000
3,wind_speed_10m,Light Gradient Boosting Machine,1.1553,2.5108,1.5843,0.8567,0.269500,0.3082,0.114000
4,wind_speed_10m,Random Forest Regressor,1.1607,2.5581,1.5990,0.8540,0.270500,0.3111,0.406000
5,wind_speed_10m,Extra Trees Regressor,1.1559,2.5908,1.6091,0.8520,0.273000,0.3128,0.220000
6,ocean_current_velocity,Extra Trees Regressor,0.1101,0.0368,0.1902,0.9693,0.092200,0.1879,0.160000
7,ocean_current_velocity,Light Gradient Boosting Machine,0.1203,0.0432,0.2060,0.9641,0.094400,0.1900,0.112000
8,ocean_current_velocity,Extreme Gradient Boosting,0.1200,0.0441,0.2085,0.9633,0.095900,0.1968,0.078000
9,sea_surface_temperature,Extra Trees Regressor,0.0396,0.0029,0.0542,0.9978,0.001800,0.0013,0.150000


In [38]:
# =====================================================================
# SEL 3: HYPERPARAMETER TUNING (Aman dari Multi-processing Windows)
# =====================================================================
from pycaret.regression import tune_model

tuned_summary_rows = []
all_tuned_models = {}

print("--- Memulai Proses Tuning Bawaan PyCaret (Single-Core Mode) ---")

# Memaksa pembatasan resource core core untuk mencegah BrokenProcessPoolException di OS Windows
set_config('n_jobs_param', 1)

for target in features:
    print(f"\n{'='*20} TUNING TARGET: {target} {'='*20}")
    
    # Mengambil kandidat algoritma peringkat ke-1
    best_model_top1 = all_results[target]["best_models"][0]
    model_name = best_model_top1.__class__.__name__
    print(f"Model Dasar: {model_name}")
    
    # Overwrite parameter internal estimator jika mendukung n_jobs paralel
    if hasattr(best_model_top1, 'n_jobs'):
        best_model_top1.set_params(n_jobs=1)
    
    try:
        # Eksekusi Randomized Search optimization
        tuned_model = tune_model(
            best_model_top1, 
            n_iter=10, 
            choose_better=True, 
            verbose=False
        )
        
        all_tuned_models[target] = tuned_model
        
        # Ekstraksi hasil metrik evaluasi setelah tuning
        tuned_results = pull()
        best_tuned_row = tuned_results.iloc[0:1].copy()
        best_tuned_row.insert(0, "Target Variable", target)
        best_tuned_row.insert(1, "Model Name", model_name)
        
        tuned_summary_rows.append(best_tuned_row)
        print(f" -> [SUKSES] R2 Akhir untuk {target}: {best_tuned_row['R2'].values[0]:.4f}")
        
    except Exception as e:
        print(f" -> [GAGAL] Optimasi {model_name} terhambat: {e}")

# --- TAMPILKAN TABEL RINGKASAN SETELAH TUNING ---
if tuned_summary_rows:
    pycaret_tuned_df = pd.concat(tuned_summary_rows, ignore_index=True)
    styled_tuned = pycaret_tuned_df.style.background_gradient(
        subset=["R2"], cmap="YlGn"
    ).background_gradient(
        subset=["MAE", "RMSE"], cmap="Reds"
    ).format({
        "R2": "{:.4f}", "MAE": "{:.4f}", "RMSE": "{:.4f}", "MSE": "{:.4f}", "MAPE": "{:.4f}"
    })
    print("\n--- TABEL RINGKASAN REKAPITULASI SETELAH TUNING ---")
    display(styled_tuned)

--- Memulai Proses Tuning Bawaan PyCaret (Single-Core Mode) ---

==================== TUNING TARGET: wave_height ====================
Model Dasar: LinearRegression
 -> [SUKSES] R2 Akhir untuk wave_height: 0.6890

==================== TUNING TARGET: wind_speed_10m ====================
Model Dasar: LGBMRegressor
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.5, subsample=1.0 will be ignored. Current value: bagging_fraction=0.5
[LightGBM] [Warning] bagging_freq is set=0, subsample_freq=0 will be ignored. Current value: bagging_freq=0
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.5, subsample=1.0 will be ignored. Current value: bagging_fraction=0.5
[LightGBM] [Warning] bagging_freq is set=0, subsample_freq=0 will be ignored. Current value: baggi

,Target Variable,Model Name,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,wave_height,LinearRegression,2213.9451,9852328.0000,3138.8418,0.6890,0.645300,0.7356
1,wind_speed_10m,LGBMRegressor,1962.9483,8507038.4842,2916.6828,0.7315,0.564300,0.5574
2,ocean_current_velocity,ExtraTreesRegressor,1989.0541,8470424.2714,2910.3993,0.7326,0.548100,0.5459
3,sea_surface_temperature,ExtraTreesRegressor,1989.0541,8470424.2714,2910.3993,0.7326,0.548100,0.5459
4,precipitation,GradientBoostingRegressor,1951.0264,8346245.9091,2888.9870,0.7365,0.542200,0.4583
5,visibility,RandomForestRegressor,1902.6444,7704155.0141,2775.6360,0.7568,0.515100,0.5004


In [3]:
# =====================================================================
# SEL 4: EVALUASI INTERAKTIF MODEL TEROPTIMASI
# =====================================================================
from pycaret.regression import evaluate_model

# FIXED: Pemanggilan model disesuaikan secara dinamis dari kamus objek all_tuned_models
print("\n=== DASHBOARD EVALUASI ITERATIF LENTERA LAUT ===")

for target in features:
    if target in all_tuned_models:
        print(f"\n📊 Membuka Dashboard Evaluasi untuk Target: {target}")
        evaluate_model(all_tuned_models[target])
    else:
        print(f"⚠️ Model teroptimasi untuk target {target} tidak tersedia.")


=== DASHBOARD EVALUASI ITERATIF LENTERA LAUT ===


NameError: name 'all_tuned_models' is not defined